In [4]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
import warnings

warnings.filterwarnings('ignore')

In [5]:
def ucb_acquisition(x, gp, kappa = 2.0):
    #Upper Confidence Bound (UCB) acquisition function
    #x = x.reshape(1, -1)
    
    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
    
    # Calculate UCB: mu + sqrt(beta) * sigma
    #ucb = mu + np.sqrt(beta) * sigma
    
    #return -ucb[0] # Return the negative UCB for minimisation
    return mu + kappa * sigma

def format_submission_string(x_best):
    x_formatted = [f"{coord:.6f}" for coord in x_best]
    return "-".join(x_formatted)

In [6]:
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy')
    
print(X)
print(Y)

[[0.66579958 0.12396913]
 [0.87779099 0.7786275 ]
 [0.14269907 0.34900513]
 [0.84527543 0.71112027]
 [0.45464714 0.29045518]
 [0.57771284 0.77197318]
 [0.43816606 0.68501826]
 [0.34174959 0.02869772]
 [0.33864816 0.21386725]
 [0.70263656 0.9265642 ]]
[ 0.53899612  0.42058624 -0.06562362  0.29399291  0.21496451  0.02310555
  0.24461934  0.03874902 -0.01385762  0.61120522]


In [7]:
X_init = np.array([
 [0.66579958, 0.12396913],
 [0.87779099, 0.7786275 ],
 [0.14269907, 0.34900513],
 [0.84527543, 0.71112027],
 [0.45464714, 0.29045518],
 [0.57771284, 0.77197318],
 [0.43816606, 0.68501826],
 [0.34174959, 0.02869772],
 [0.33864816, 0.21386725],
 [0.70263656, 0.9265642 ]]
)

y_init = np.array(
[ 0.53899612, 0.42058624, -0.06562362,  0.29399291,  0.21496451,  0.02310555,
  0.24461934, 0.03874902, -0.01385762,  0.61120522]
)

y_trans = y_init.copy()

kernel = Matern(length_scale=0.1, length_scale_bounds=(1e-1, 1e2), nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, random_state=42)

gp.fit(X_init, y_trans)

grid_size = 100
x1 = np.linspace(0, 1, grid_size)
x2 = np.linspace(0, 1, grid_size)
X_candidates = np.array([[i, j] for i in x1 for j in x2])

# --- 5. Evaluate acquisition function ---
acq_values = ucb_acquisition(X_candidates, gp, kappa=2.5)

# --- 6. Select next point ---
next_point = X_candidates[np.argmax(acq_values)]

print(format_submission_string(next_point))
#print(f"Next point to query:{next_point:.6f}")

0.000000-1.000000
